# Using Roboflow Models to Process Real Data

This notebook is meant to show you how to deploy the nano model **you just trained** on **new images** using Python code.

*Don't worry too much here about the code itself* if you are new to python or coding.

The main goal here are to help you visualize all the the pieces of the puzzle, and see how they can fit together.

**For your own research, I highly recommend:**

*   Storing any scripts you develop in a github repository
*   Working within a virtual environment on your own computer (or server) with all the packages you need in one place
*   Using Claude or other LLMs to help generate/debug code if needed, but beware: **it tends to over-engineer basic steps!**
*   Recycling code from this Colab :)





## PART A: Using Your Flower Visitor Detection Model

### Step 1: Set-Up

First, you will also need an API KEY to access your model which now lives in Roboflow's server:

- Go to your roboflow profile
- Click '⚙️ Settings' on the sidebar
- Click '🔑 API Keys' from the menu
- Click the copy button to add the key to your clipboard

**As with any API key, KEEP THIS PRIVATE!**

We will use colab's secret key feature so we don't hard-code our keys in.
- In colab, click the 🔑 icon
- Add new secret key
- Name it ROBOFLOW_API_KEY
- Paste your API Key in
- Click the toggle for notebook access (make sure it is a blue check mark)

Then, run the two chunks of code below:


In [ ]:
# This installs roboflow's python package

!pip install roboflow -q


In [ ]:
import os
import json
import glob
import csv
from roboflow import Roboflow
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from google.colab import userdata

# Reads your key from Colab's Secrets manager (the key icon in the left sidebar).
ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

print("✅ Using Google Colab secrets for Roboflow API Key:\n",ROBOFLOW_API_KEY)


Every Roboflow model lives at a specific **workspace / project / version**. Have your model page open so you can copy and paste these in.

- **Workspace**: shown in your Roboflow project's URL, right after `app.roboflow.com/`
- **Project**: copy-pasteable Project ID (the name of your model!)
- **Version**: the version number of the training run you just finished


In [ ]:
# FILL IN: these three values from your own Roboflow project
## or use workspace "field-museum", project "flow-vis-detect", version 3

WORKSPACE = "YOUR-WORKSPACE-HERE"
PROJECT = "YOUR-MODEL-HERE"
VERSION = 1

# These lines of code make sure your models are reachable and trained
version = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION)
trained_models = version.models()
model = trained_models[0]
print("Model connected!")


Then, run this to get some new images the model hasn't seen yet. If you were running this code locally in your terminal, for example, you would need to set the path to whatever folder has your images. For now we can get the images from the workshop's github.

In [ ]:
# This gets folders of images I've already collected and defines a path to them.

REPO_URL = "https://github.com/PrairieResearchInstitute/2026_ml_eco_evo_collections_workshop.git"
REPO_DIR = "2026_ml_eco_evo_collections_workshop"

if not os.path.isdir(REPO_DIR):
    os.system(f"git clone --depth 1 {REPO_URL}")

DETECTION_IMAGES_DIR = os.path.join(REPO_DIR, "computer_vision", "goldenrod_visitors", "new_images")

### Step 2: Run Model on ONE new image

First let's look at a single test image:

In [ ]:
TEST_IMAGE_PATH = sorted(glob.glob(os.path.join(DETECTION_IMAGES_DIR, "*.jpg")))[0]

img = Image.open(TEST_IMAGE_PATH)
plt.imshow(img)
plt.axis("off")
plt.title("New Image 1")
plt.show()


Now let's send that image to our model. We can use confidence and overlap (0-100) to tune our results.

**Confidence:** lower for more bounding boxes that the model is less confident about (lower confidence = more false positives, but also fewer "misses").

**Overlap:** adjust higher or lower depending on how much you expect object bounding boxes to overlap. In this image, overlap might be pretty high since the flies are clustered together.

Try changing the confidence settings to see how it affects the number of predictions your model returns.

In [ ]:
# Run the model on the image. The ".json" part means we'll save the prediction in JSON format.

result = model.predict(TEST_IMAGE_PATH, confidence=25, overlap=75).json() # try changing the confidence!

result


Each item in `result["predictions"]` is one detected object: an `x, y` center point, a `width`/`height`, a `class` name, and a `confidence` score.


In [ ]:
print(f"Found {len(result['predictions'])} objects")
for pred in result["predictions"]:
    print(f"  {pred['class']}  (confidence: {pred['confidence']:.2f})")

# Save the raw result so we have a record of it
with open("single_image_result.json", "w") as f:
    json.dump(result, f, indent=2)

print("Saved to single_image_result.json")


Now let's draw a bounding box. Roboflow gives us the **center point** (`x`, `y`) and the box's `width`/`height` — but matplotlib wants the **top-left corner**. So first, just one box, drawn by hand, to see the math:


In [ ]:
# Grab just the first prediction to start with, which starts at "0"
first_pred = result["predictions"][0]

# Convert center-point + size into a top-left corner, for matplotlib
box_left = first_pred["x"] - first_pred["width"] / 2
box_top = first_pred["y"] - first_pred["height"] / 2

fig, ax = plt.subplots()
ax.imshow(img)

box = patches.Rectangle(
    (box_left, box_top),
    first_pred["width"],
    first_pred["height"],
    linewidth=2,
    edgecolor="red",
    facecolor="none",
)
ax.add_patch(box)
ax.set_title(f"{first_pred['class']} ({first_pred['confidence']:.2f})")
ax.axis("off")
plt.show()


That works for one box. Now let's do the same thing for **every** prediction, in a step that we define as "draw_detections"


In [ ]:
def draw_detections(image_path, predictions, ax):
    """Draw every bounding box + label onto a matplotlib axis."""
    img = Image.open(image_path)
    ax.imshow(img)

    for pred in predictions:
        box_left = pred["x"] - pred["width"] / 2
        box_top = pred["y"] - pred["height"] / 2

        box = patches.Rectangle(
            (box_left, box_top),
            pred["width"],
            pred["height"],
            linewidth=2,
            edgecolor="red",
            facecolor="none",
        )
        ax.add_patch(box)
        ax.text(
            box_left, box_top - 5,
            f"{pred['class']} ({pred['confidence']:.2f})",
            color="red", fontsize=8,
        )

    ax.axis("off")

fig, ax = plt.subplots(figsize=(8, 8))
draw_detections(TEST_IMAGE_PATH, result["predictions"], ax)
plt.show()


### Step 3: Multiple images

Same steps as Part 2, just looped over many images instead of one.


In [ ]:
# Here we define our path as ALL .jpg images in the folder of new images (there are 5)

image_paths = glob.glob(os.path.join(DETECTION_IMAGES_DIR, "*.jpg"))
print(f"Found {len(image_paths)} images")


Loop over every image, run the model, and keep every result. Then we can


In [ ]:
all_results = {}  # image path -> prediction result

for path in image_paths:
    result = model.predict(path, confidence=30, overlap=50).json()

    all_results[path] = result
    print(f"{os.path.basename(path)}: {len(result['predictions'])} objects found")

# Save every result to one combined JSON file
with open("batch_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("Saved to batch_results.json")

# we can also visualize what the results look like

N_TO_SHOW = 4
sample_paths = image_paths[:N_TO_SHOW]

fig, axes = plt.subplots(1, 4, figsize=(15, 5))

for path, ax in zip(sample_paths, axes.flatten()):
    predictions = all_results[path]["predictions"]
    draw_detections(path, predictions, ax)
    ax.set_title(os.path.basename(path), fontsize=9)

plt.tight_layout()
plt.show()

We can also use these coordinates to **crop out detections**, so each detection gets its own image.

This is often a very useful step to do for images where you want to classify different things from an image, e.g., the abudance and diversity of insect visitors on flowers.

You could even chain more models together, for example, **detection** -> **segmentation** to remove the background -> **classification** to get taxonomic identities without noisy backgrounds (or shortcut learning)

In [ ]:
CROPS_DIR = "detection_crops"
os.makedirs(CROPS_DIR, exist_ok=True)

crop_paths = []

for path, result in all_results.items():
    img = Image.open(path)
    base_name = os.path.splitext(os.path.basename(path))[0]

    for i, pred in enumerate(result["predictions"]):
        box_left = pred["x"] - pred["width"] / 2
        box_top = pred["y"] - pred["height"] / 2
        box_right = pred["x"] + pred["width"] / 2
        box_bottom = pred["y"] + pred["height"] / 2

        crop = img.crop((box_left, box_top, box_right, box_bottom))
        crop_path = os.path.join(CROPS_DIR, f"{base_name}_crop{i}.jpg")
        crop.save(crop_path)
        crop_paths.append(crop_path)

print(f"Saved {len(crop_paths)} cropped detections -> {CROPS_DIR}")

In [ ]:
# check out our crops!
N_CROPS_TO_SHOW = 3
sample_crop_paths = crop_paths[:N_CROPS_TO_SHOW]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for crop_path, ax in zip(sample_crop_paths, axes.flatten()):
    crop_img = Image.open(crop_path)
    ax.imshow(crop_img)
    ax.set_title(os.path.basename(crop_path), fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

### Step 4: Save Our Data as a CSV

We can summarize the data in a CSV, which we can use for downstream analyses like counting the number of detections per image.

In [ ]:
from collections import Counter

count_rows = []
for path, result in all_results.items():
    class_counts = Counter(pred["class"] for pred in result["predictions"])
    for class_name, count in class_counts.items():
        count_rows.append({
            "image": os.path.basename(path),
            "class": class_name,
            "count": count,
        })

with open("class_counts.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["image", "class", "count"])
    writer.writeheader()
    writer.writerows(count_rows)

for row in count_rows:
    print(row)


##Part B: Linking Lots of Models Together

To show you what a multi-model pipeline looks like in action, we can build a workflow that uses two models from my collections digitization pipeline, *DrawerDissect*: https://github.com/EGPostema/DrawerDissect

This will use all the principles we saw in Part A. Setting up our model, setting paths to all images in a given folder, looping through the images to detect and crop, and writing the data in CSV format. Instead of seperating these steps out I've combined them into bigger, more comprehensive "chunks", and I've also added in segmentation so we can measure objects!

###Step 1: Set-Up

In [ ]:
!pip install roboflow opencv-python-headless pandas matplotlib -q

In [ ]:
import os
import glob
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from roboflow import Roboflow
from google.colab import userdata
Image.MAX_IMAGE_PIXELS = None # lets us open BIG images

ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

# Use the FMNH's public workspace instead:
ROBOFLOW_WORKSPACE = "field-museum"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
workspace = rf.workspace(ROBOFLOW_WORKSPACE)

# Set up our models; one for detection, one for segmentation

DETECTION_ENDPOINT = "bugfinder-kdn9e"
DETECTION_VERSION = 21
DETECTION_CONFIDENCE = 50

SEGMENTATION_ENDPOINT = "bugmasker-all"
SEGMENTATION_VERSION = 14
SEGMENTATION_CONFIDENCE = 50

detection_model = workspace.project(DETECTION_ENDPOINT).version(DETECTION_VERSION).models()[0]
segmentation_model = workspace.project(SEGMENTATION_ENDPOINT).version(SEGMENTATION_VERSION).models()[0]

# A folder to keep everything organized as we go.
OUTPUT_DIR = "dissect_demo_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Make a path to the tray images
TRAYS_DIR = os.path.join(REPO_DIR, "computer_vision", "cicindela_trays")
TRAY_IMAGE_PATHS = sorted(glob.glob(os.path.join(TRAYS_DIR, "*.jpg")))
print(f"Found {len(TRAY_IMAGE_PATHS)} tray images")

# We can see what kind of image we're working with below (optional):

sample_tray = Image.open(TRAY_IMAGE_PATHS[0])
plt.imshow(sample_tray)
plt.axis("off")
plt.title(os.path.basename(TRAY_IMAGE_PATHS[0]))
plt.show()

###Step 2: Detect, Segment, Measure Loop

For each tray we want to **detect** every specimen, **crop** it out of the image, **segment** the crop to get its outline, **measure** that outline, and keep a **running table** of every specimen measured.

In [ ]:
rows = []

for tray_path in TRAY_IMAGE_PATHS:
    tray_id = os.path.splitext(os.path.basename(tray_path))[0]

    tray_img = Image.open(tray_path)
    if tray_img.mode in ("RGBA", "LA", "P"):
        tray_img = tray_img.convert("RGB")

    # DETECT every specimen in the tray
    detection = detection_model.predict(tray_path, confidence=DETECTION_CONFIDENCE, overlap=50).json()
    print(f"{tray_id}: found {len(detection['predictions'])} specimens")

    for i, det in enumerate(detection["predictions"], start=1):
        specimen_id = f"{tray_id}_specimen_{i:03}"

        # Crop the specimen straight out of the tray
        x, y, w, h = det["x"], det["y"], det["width"], det["height"]
        crop = tray_img.crop((x - w / 2, y - h / 2, x + w / 2, y + h / 2))
        crop_path = os.path.join(OUTPUT_DIR, f"{specimen_id}.jpg")
        crop.save(crop_path)

        # SEGMENT the cropped specimen to get its outline
        segmentation = segmentation_model.predict(crop_path, confidence=SEGMENTATION_CONFIDENCE).json()
        if not segmentation["predictions"]:
            continue  # no outline found for this crop -- skip measuring it

        points = segmentation["predictions"][0]["points"]
        polygon = [(int(p["x"]), int(p["y"])) for p in points]

        # Turn the outline into a mask, then MEASURE it
        mask = Image.new("L", crop.size)
        ImageDraw.Draw(mask).polygon(polygon, outline=0, fill=255)
        mask_array = cv2.threshold(np.array(mask), 127, 255, cv2.THRESH_BINARY)[1]
        contours, _ = cv2.findContours(mask_array, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            continue

        contour = max(contours, key=cv2.contourArea)
        area_px = cv2.contourArea(contour)
        (_, _), (mw, mh), _ = cv2.minAreaRect(contour)

        rows.append({
            "specimen_id": specimen_id,
            "tray": tray_id,
            "length_px": max(mw, mh),
            "width_px": min(mw, mh),
            "area_px": area_px,
        })

measurements_df = pd.DataFrame(rows)
measurements_df.to_csv(os.path.join(OUTPUT_DIR, "measurements.csv"), index=False)
print(f"\nMeasured {len(measurements_df)} specimens -> measurements.csv")
measurements_df.head()

###Step 3: Visualize

In [ ]:
# Let's look at some examples of our cropped, measured specimens

N_TO_SHOW = 4
sample_ids = measurements_df["specimen_id"].head(N_TO_SHOW).tolist()

fig, axes = plt.subplots(1, len(sample_ids), figsize=(5 * len(sample_ids), 5))
if len(sample_ids) == 1:
    axes = [axes]

for specimen_id, ax in zip(sample_ids, axes):
    crop_path = os.path.join(OUTPUT_DIR, f"{specimen_id}.jpg")
    ax.imshow(Image.open(crop_path))
    row = measurements_df[measurements_df["specimen_id"] == specimen_id].iloc[0]
    ax.set_title(f"{specimen_id}\nlength={row['length_px']:.0f}px", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Finally, we can get a sense of the distribution of sizes for each species

# The tray filename itself is the species (e.g. "latesignata.jpg")
measurements_df["species"] = measurements_df["tray"]

# Area is a squared unit, so the px-to-mm conversion factor needs to be
# squared too -- dividing by PIXELS_PER_MM alone (not squared) is a common
# mistake and gives numbers that look plausible but are wrong.
PIXELS_PER_MM = 106
measurements_df["area_mm2"] = measurements_df["area_px"] / (PIXELS_PER_MM ** 2)

fig, ax = plt.subplots(figsize=(8, 6))
measurements_df.boxplot(column="area_mm2", by="species", ax=ax)
plt.suptitle("")  # clear pandas' default auto-title, keep just our own below
ax.set_xlabel("Species")
ax.set_ylabel("Body area (mm²)")
ax.set_title("Variation in Body Size by Species")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

###**BONUS: Make Backgroundless Images**

These are useful for things like extracting color patterns, where you don't want any pixels from the background. To do this, we use the segmentation coordinates to turn anything outside the "body" polygon transparent.

They are also nice to add to figures, which we can do now...

In [ ]:
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

def make_transparent(crop, mask):
    """Return an RGBA version of crop where everything outside the mask is transparent."""
    rgba = crop.convert("RGBA")
    rgba.putalpha(mask)
    return rgba

# Get one representative specimen (crop + mask) per tray
tray_representative = {}  # tray_id -> {"crop", "mask"}

for tray_path in TRAY_IMAGE_PATHS:
    tray_id = os.path.splitext(os.path.basename(tray_path))[0]

    tray_img = Image.open(tray_path)
    if tray_img.mode in ("RGBA", "LA", "P"):
        tray_img = tray_img.convert("RGB")

    detection = detection_model.predict(tray_path, confidence=DETECTION_CONFIDENCE, overlap=50).json()
    if not detection["predictions"]:
        continue

    det = detection["predictions"][0]  # just the first specimen found in this tray
    x, y, w, h = det["x"], det["y"], det["width"], det["height"]
    crop = tray_img.crop((x - w / 2, y - h / 2, x + w / 2, y + h / 2))
    crop_path = os.path.join(OUTPUT_DIR, f"{tray_id}_representative.jpg")
    crop.save(crop_path)

    segmentation = segmentation_model.predict(crop_path, confidence=SEGMENTATION_CONFIDENCE).json()
    if not segmentation["predictions"]:
        continue

    points = segmentation["predictions"][0]["points"]
    polygon = [(int(p["x"]), int(p["y"])) for p in points]

    mask = Image.new("L", crop.size)
    ImageDraw.Draw(mask).polygon(polygon, outline=0, fill=255)

    tray_representative[tray_id] = {"crop": crop, "mask": mask}

fig, ax = plt.subplots(figsize=(8, 6))
measurements_df.boxplot(column="area_mm2", by="species", ax=ax, grid=False)
plt.suptitle("")
ax.set_xlabel("Species")
ax.set_ylabel("Body area (mm²)")
ax.set_title("Specimen body area by species")
plt.xticks(rotation=45, ha="right")

species_order = sorted(measurements_df["species"].unique())
y_top = measurements_df["area_mm2"].max()
ax.set_ylim(top=y_top * 1.3)

THUMBNAIL_ZOOM = 0.05

for i, species in enumerate(species_order, start=1):
    if species not in tray_representative:
        continue
    rep = tray_representative[species]
    thumb = make_transparent(rep["crop"], rep["mask"])
    imagebox = OffsetImage(thumb, zoom=THUMBNAIL_ZOOM)
    ab = AnnotationBbox(imagebox, (i, y_top * 1.1), frameon=False)
    ax.add_artist(ab)

plt.tight_layout()
plt.show()

# Citations

---

## Cite This Demo

If you use or adapt this material, please cite:

> Elizabeth Postema. (2026). *Coding Demo for UIUC ML Workshop*. [Workshop teaching materials]. Presented at Machine Learning for Ecology and Evolution Research Using Natural History Collections, University of Illinois Urbana-Champaign. https://github.com/PrairieResearchInstitute/2026_ml_eco_evo_collections_workshop

---

## Read the Pre-Print for DrawerDissect!
> Postema, E.G., Briscoe, L., Harder, C., Hancock, G.R.A., Guarnieri, L.D, Eisel, T., Welch, K., Fisher, N., Johnson, C., Souza, D., Sepulveda, T., Phillip, D., Baquiran, R., de Medeiros, B.A.S. 2025. DrawerDissect: Whole-drawer insect imaging, segmentation, and trait extraction using AI. EcoEvoRxiv (pre-print). https://doi.org/10.32942/X2QW84

  ## AI Disclosure

  Much of the code used in this tutorial was generated using a Claude LLM; all text was designed/written by E.G. Postema.